In [ ]:
#import Library
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re

In [3]:
#requesting the webpage
url = "https://books.toscrape.com/"
session=requests.session()
response = requests.get(url)
print(response.status_code)

200


In [ ]:
soup= BeautifulSoup(response.content, "html.parser")
titles, price_excl_taxes, price_incl_taxes, taxes, stock_counts, scrape_timestamps, upc_numbers, product_types, num_reviews, star_ratings, descriptions, image_urls, product_page_urls, availabilities, ratings, categories = [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []

# Make sure images folder exists
os.makedirs(r"C:\Users\JAY LODHA\web-scraping-project\data\raw\images", exist_ok=True)

In [ ]:
page=1
while True:
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = session.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve page {page}. Status code: {response.status_code}")
        break
    soup = BeautifulSoup(response.content, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    if not books:
        print(f"No books found on page {page}.")
        break

    for book in books:

        title_link = book.find("div", class_="image_container").find("a")["href"]

        page_link = "https://books.toscrape.com/catalogue/" + title_link
        page_response = session.get(page_link)
        page_soup = BeautifulSoup(page_response.content, "html.parser")

        title = page_soup.find("div", class_="product_main").find("h1").text
        titles.append(title)

        price_excl_tax = page_soup.find("table", class_="table table-striped").find_all("tr")[2].find("td").text
        price_excl_taxes.append(price_excl_tax)

        price_incl_tax = page_soup.find("table", class_="table table-striped").find_all("tr")[3].find("td").text
        price_incl_taxes.append(price_incl_tax)

        availability = page_soup.find("table", class_="table table-striped").find_all("tr")[5].find("td").text
        availabilities.append(availability)

        rating = page_soup.find("p", class_="star-rating")["class"][1]
        ratings.append(rating)

        tax = page_soup.find("table",class_="table table-striped").find_all("tr")[4].td.text
        taxes.append(tax)

        upc = page_soup.find("table",class_="table table-striped").find_all("tr")[0].td.text
        upc_numbers.append(upc)

        product_type = page_soup.find("table",class_="table table-striped").find_all("tr")[1].td.text
        product_types.append(product_type)

        num_review = page_soup.find("table",class_="table table-striped").find_all("tr")[6].td.text
        num_reviews.append(num_review)

        desc_tag = page_soup.find("div", id="product_description")
        if desc_tag:
            description = desc_tag.find_next_sibling("p").text
        else:
            description = "No Description Available"
        descriptions.append(description)

        product_page_url = page_link
        product_page_urls.append(product_page_url)

        stock_count = page_soup.find("table", class_="table table-striped").find_all("tr")[5].find("td").text[10:12]
        stock_counts.append(stock_count)

        scrape_timestamp = pd.Timestamp.now().strftime("%Y-%m-%dT%H:%M:%S")
        scrape_timestamps.append(scrape_timestamp)

        category = page_soup.find("ul",class_="breadcrumb").find_all("li")[2].a.text
        categories.append(category)

        # Image download
        image_url = "https://books.toscrape.com/" + page_soup.find("div", class_="item active").find("img")["src"].replace("../", "")
        safe_title = re.sub(r'[\\/*?:"<>|]', "", title)   # remove invalid chars
        safe_title = safe_title.replace(" ", "_")         # replace spaces with underscores

        img_filename = rf"C:\Users\JAY LODHA\web-scraping-project\data\raw\images\{safe_title}.jpg"
        img_response = session.get(image_url)
        with open(img_filename, "wb") as f:
            f.write(img_response.content)

        
        image_urls.append(f"{safe_title}.jpg")


    print(f"Page {page} scraped successfully.")
    page += 1

data = pd.DataFrame({
    "Title": titles,
    "Price (Excl Tax)": price_excl_taxes,
    "Price (Incl Tax)": price_incl_taxes,
    "Tax": taxes,
    "UPC": upc_numbers,
    "Product Type": product_types,
    "Num Reviews": num_reviews,
    "Description": descriptions,
    "Product Page URL": product_page_urls,
    "Availability": availabilities,
    "Rating": ratings,
    "Category": categories,
    "Image URL": image_urls,
    "Stock Count": stock_counts,
    "Scrape Timestamp": scrape_timestamps
})
data['Image URL'].isnull().value_counts()

In [ ]:
data.to_csv(r"CSV\books_phase_3_updates.csv", index=False, encoding="utf-8-sig")